# JiT-RCDM Training on Google Colab

Representation-Conditioned Diffusion Model — DinoV3 ViT-S/16 + JiT ViT + Flow Matching  
Dataset: Messidor-2 retinal fundus images (224×224)

## Before you start
1. **Runtime → Change runtime type → A100 GPU**
2. Upload the following to **Google Drive** under `MyDrive/jit_rcdm/`:
   - `train_reps.pt` — precomputed DinoV3 representations
   - `dinov3_vits16_tmp/` — DinoV3 checkpoint folder (`config.json` + `model.safetensors`)
   - `test_images/` — a few `.png` fundus images for sampling
3. Run cells **top to bottom**

## Workflow after local code changes
Edit locally → `git push` → re-run **Cell 3** in Colab → re-run **Cell 7** to reload imports.

## 1 — Check GPU

In [1]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM    : {mem:.1f} GB")
    if mem >= 70:
        print("✓ A100/H100 80GB — use batch_size=256")
    elif mem >= 38:
        print("✓ A100 40GB — use batch_size=128")
    else:
        print("⚠ T4/V100 — use batch_size=32")
else:
    print("❌ No GPU — go to Runtime → Change runtime type → GPU")

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : NVIDIA A100-SXM4-40GB
VRAM    : 42.4 GB
✓ A100 40GB — use batch_size=128


## 2 — Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = "/content/drive/MyDrive/jit_rcdm"
os.makedirs(DRIVE_DIR, exist_ok=True)

print(f"Drive mounted. Project folder: {DRIVE_DIR}")
print("Contents:")
for f in sorted(os.listdir(DRIVE_DIR)):
    full = os.path.join(DRIVE_DIR, f)
    tag  = f"({os.path.getsize(full)/1e6:.1f} MB)" if os.path.isfile(full) else "(folder)"
    print(f"  {f} {tag}")

Mounted at /content/drive
Drive mounted. Project folder: /content/drive/MyDrive/jit_rcdm
Contents:
  checkpoints (folder)
  data (folder)
  dinov3_vits16_tmp (folder)
  test_images (folder)
  train_packed.pt (147.8 MB)
  train_reps.pt (1.6 MB)


## 3 — Clone / pull JiT-RCDM code

> **After a local `git push`:** re-run this cell to pull the latest changes, then re-run Cell 7 to reload imports.

In [4]:
import os, sys

REPO_DIR = "/content/jit_rcdm"
REPO_URL = "https://github.com/SeverinLe/master_implementation.git"
BRANCH   = "claude/silly-faraday-d8512b"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already cloned — pulling latest changes...")
    !git -C {REPO_DIR} pull
else:
    print(f"Cloning branch {BRANCH} ...")
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

%cd {REPO_DIR}

# Verify JiT code is present
assert os.path.exists("rcdm/jit.py"),       "❌ rcdm/jit.py missing — git pull may have failed"
assert os.path.exists("scripts/train.py"),  "❌ scripts/train.py missing"
print("\n✓ JiT-RCDM code present")
!ls rcdm/

Repo already cloned — pulling latest changes...
Already up to date.
/content/jit_rcdm

✓ JiT-RCDM code present
conditioning.py  encoder.py   jit.py
dataset.py	 __init__.py  test_conditioning.py


## 4 — Install dependencies

In [5]:
!pip install -q transformers safetensors wandb
print("✓ Dependencies installed")

from transformers import AutoModel
print("✓ transformers import OK (PyTorch >= 2.4 confirmed)")

✓ Dependencies installed
✓ transformers import OK (PyTorch >= 2.4 confirmed)


## 5 — Log in to Weights & Biases

**One-time setup — add your API key as a Colab secret:**
1. Click the 🔑 **key icon** in the left sidebar
2. Click **+ Add new secret**
3. Name: `WANDB_API_KEY` — Value: your key from [wandb.ai/authorize](https://wandb.ai/authorize)
4. Toggle **Notebook access** ON
5. Re-run this cell

In [6]:
import wandb
from google.colab import userdata

WANDB_KEY = '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'
assert WANDB_KEY, (
    "WANDB_API_KEY secret not found.\n"
    "Follow the steps in the cell above, then re-run."
)

wandb.login(key=WANDB_KEY, relogin=True)
print(f"✓ W&B logged in as: {wandb.api.default_entity}")
print(f"  Project URL: https://wandb.ai/{wandb.api.default_entity}/jit-rcdm")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: severinle (severinle-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ W&B logged in as: severinle-johannes-kepler-universit-t-linz
  Project URL: https://wandb.ai/severinle-johannes-kepler-universit-t-linz/jit-rcdm


## 6 — Copy DinoV3 checkpoint from Drive

In [7]:
import shutil, os, json

SRC = "/content/drive/MyDrive/jit_rcdm/dinov3_vits16_tmp"
DST = "/content/jit_rcdm/checkpoints/dinov3_vits16_tmp"

assert os.path.isdir(SRC), (
    f"DinoV3 checkpoint not found at {SRC}.\n"
    "Upload dinov3_vits16_tmp/ to Google Drive at MyDrive/jit_rcdm/"
)

if not os.path.isdir(DST):
    shutil.copytree(SRC, DST)
    print(f"✓ Copied DinoV3 checkpoint → {DST}")
else:
    print(f"✓ DinoV3 checkpoint already at {DST}")

cfg = json.load(open(os.path.join(DST, "config.json")))
assert cfg['use_gated_mlp'] == False, "use_gated_mlp must be false — check config.json"
assert cfg['hidden_size']   == 384,   "Expected hidden_size=384"
print(f"  hidden_size={cfg['hidden_size']}, use_gated_mlp={cfg['use_gated_mlp']}, "
      f"num_register_tokens={cfg['num_register_tokens']}")
print("✓ Config OK")

✓ Copied DinoV3 checkpoint → /content/jit_rcdm/checkpoints/dinov3_vits16_tmp
  hidden_size=384, use_gated_mlp=False, num_register_tokens=4
✓ Config OK


## 7 — Copy packed dataset from Drive

`train_packed.pt` stores both image tensors (uint8) **and** DinoV3 representations in one file — no raw image folder needed on Colab.

**Create it locally first (run once on your Mac):**
```bash
python .claude/worktrees/silly-faraday-d8512b/scripts/pack_dataset.py \
    --reps_file  data/messidor2/train_reps.pt \
    --out_file   data/messidor2/train_packed.pt \
    --image_size 224
```
Then upload `data/messidor2/train_packed.pt` (~150 MB) to Drive at `MyDrive/jit_rcdm/train_packed.pt`.

In [8]:
import shutil, os, torch

SRC = "/content/drive/MyDrive/jit_rcdm/train_packed.pt"
DST = "/content/jit_rcdm/data/messidor2/train_packed.pt"

assert os.path.exists(SRC), (
    f"train_packed.pt not found at {SRC}.\n"
    "Run pack_dataset.py locally first, then upload to Google Drive."
)

os.makedirs(os.path.dirname(DST), exist_ok=True)
if not os.path.exists(DST):
    print("Copying train_packed.pt from Drive (~150 MB)...")
    shutil.copy2(SRC, DST)
else:
    print("train_packed.pt already present")

data = torch.load(DST, map_location="cpu", weights_only=False)
assert "images" in data and "reps" in data, "File missing 'images' or 'reps' key — re-run pack_dataset.py"
imgs, reps = data["images"], data["reps"]
assert reps.shape[1] == 384,  f"Expected 384-dim reps, got {reps.shape[1]}"
assert imgs.dtype == torch.uint8, f"Expected uint8 images, got {imgs.dtype}"
print(f"✓ {len(imgs)} packed images {tuple(imgs.shape)}, dtype={imgs.dtype}")
print(f"  reps {tuple(reps.shape)}, mean norm={reps.norm(dim=1).mean():.2f}")
print("✓ Packed dataset OK")

Copying train_packed.pt from Drive (~150 MB)...
✓ 972 packed images (972, 3, 224, 224), dtype=torch.uint8
  reps (972, 384), mean norm=10.01
✓ Packed dataset OK


## 8 — Verify full pipeline

> Re-run this cell after a `git pull` to reload updated code.

In [9]:
# Reload modules cleanly after a git pull
import importlib, sys
for mod in list(sys.modules.keys()):
    if mod.startswith("rcdm"):
        del sys.modules[mod]

import sys, os
sys.path.insert(0, "/content/jit_rcdm")

from rcdm.encoder      import load_encoder, build_transform, DINOV3_CHECKPOINT
from rcdm.jit          import JiT_S_16, FlowMatching, create_jit_model
from rcdm.dataset      import RepresentationDataset
from rcdm.conditioning import RMSNorm, AdaLNZero, ConditioningProjector
import torch

print("✓ All imports OK")
print(f"  DINOV3_CHECKPOINT → {DINOV3_CHECKPOINT}")
assert os.path.isdir(DINOV3_CHECKPOINT), f"Checkpoint not found at {DINOV3_CHECKPOINT}"
print("  Checkpoint directory exists ✓")

# Forward-pass smoke test (CPU)
m = JiT_S_16(image_size=224, h_dim=384)
m.eval()
with torch.no_grad():
    out = m(torch.randn(2,3,224,224), torch.rand(2), torch.randn(2,384))
assert out.shape == (2,3,224,224)
n = sum(p.numel() for p in m.parameters())
print(f"✓ JiT_S_16 forward pass — output {tuple(out.shape)}, {n/1e6:.1f}M params")
print(f"  null_h  : shape={tuple(m.null_h.shape)}, requires_grad={m.null_h.requires_grad}")
print(f"  freqs_cis: shape={tuple(m.freqs_cis.shape)}, dtype={m.freqs_cis.dtype}")
print("\n✓ Pipeline ready — proceed to training")

✓ All imports OK
  DINOV3_CHECKPOINT → /content/jit_rcdm/checkpoints/dinov3_vits16_tmp
  Checkpoint directory exists ✓
✓ JiT_S_16 forward pass — output (2, 3, 224, 224), 25.7M params
  null_h  : shape=(384,), requires_grad=True
  freqs_cis: shape=(196, 32), dtype=torch.complex64

✓ Pipeline ready — proceed to training


## 9 — Train

Checkpoints saved to Drive every 5 000 steps — survive session disconnects.

In [10]:
import torch
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
batch  = 256 if mem_gb >= 70 else (128 if mem_gb >= 38 else 32)
print(f"GPU VRAM: {mem_gb:.0f} GB → batch_size={batch}")

GPU VRAM: 42 GB → batch_size=128


In [22]:
import os
CKPT_DIR = "/content/drive/MyDrive/jit_rcdm/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

!python scripts/train.py \
    --model          S16 \
    --reps_file      data/messidor2/train_packed.pt \
    --save_dir       {CKPT_DIR} \
    --image_size     224 \
    --batch_size     {batch} \
    --grad_accum     1 \
    --lr             1e-4 \
    --warmup_steps   2000 \
    --total_steps    100000 \
    --save_interval  5000 \
    --log_interval   100 \
    --cfg_dropout    0.1 \
    --ema_decay      0.9999 \
    --wandb_project  jit-rcdm \
    --wandb_run_name S16-100k-A100 \
    --device         cuda


[1/4] Building JiT model...
  Using preset JiT_S16
  Total parameters  : 25.7M
  Trainable         : 25.7M

[2/4] Loading dataset...
Loading representations from data/messidor2/train_packed.pt...
  972 packed image-representation pairs loaded (no disk access at training time)
  972 samples, 7 batches/epoch at batch_size=128

[3/4] Setting up optimiser...
  EMA initialised (decay=0.9999)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: severinle (severinle-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /content/jit_rcdm/wandb/run-20260516_210537-c48aue7x
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run S16-100k-A100
wandb: ⭐️ View project at https://wandb.

: 

## 9b — Resume after disconnection

Re-run cells 1–8, then run this cell. Latest checkpoint is picked up automatically.

In [11]:
import os, glob, torch
CKPT_DIR = "/content/drive/MyDrive/jit_rcdm/checkpoints"
mem_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
batch    = 256 if mem_gb >= 70 else (128 if mem_gb >= 38 else 32)

ckpts  = sorted(glob.glob(os.path.join(CKPT_DIR, "jit_rcdm_step*.pt")))
assert ckpts, f"No checkpoint found in {CKPT_DIR}"
latest = ckpts[-1]
print(f"Resuming from: {latest}")

!python scripts/train.py \
    --model          S16 \
    --reps_file      data/messidor2/train_packed.pt \
    --save_dir       {CKPT_DIR} \
    --image_size     224 \
    --batch_size     {batch} \
    --grad_accum     1 \
    --lr             1e-4 \
    --warmup_steps   2000 \
    --total_steps    100000 \
    --save_interval  5000 \
    --log_interval   100 \
    --cfg_dropout    0.1 \
    --ema_decay      0.9999 \
    --resume         {latest} \
    --wandb_project  jit-rcdm \
    --wandb_run_name S16-100k-A100 \
    --device         cuda

Resuming from: /content/drive/MyDrive/jit_rcdm/checkpoints/jit_rcdm_step0055000.pt

[1/4] Building JiT model...
  Using preset JiT_S16
  Resuming from /content/drive/MyDrive/jit_rcdm/checkpoints/jit_rcdm_step0055000.pt
  Resuming from step 55000
  Total parameters  : 25.7M
  Trainable         : 25.7M

[2/4] Loading dataset...
Loading representations from data/messidor2/train_packed.pt...
  972 packed image-representation pairs loaded (no disk access at training time)
  972 samples, 7 batches/epoch at batch_size=128

[3/4] Setting up optimiser...
  EMA restored (decay=0.9999)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: severinle (severinle-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is 

## 10 — Sample

| Steps trained | `--cfg_scale` |
|---|---|
| < 15 k | 1.0 |
| 15–30 k | 1.5 |
| 30–50 k | 2.0 |
| > 50 k | 2.0–3.0 |

In [12]:
import os, glob
CKPT_DIR   = "/content/drive/MyDrive/jit_rcdm/checkpoints"
SAMPLE_DIR = "/content/drive/MyDrive/jit_rcdm/samples"
os.makedirs(SAMPLE_DIR, exist_ok=True)

ckpts  = sorted(glob.glob(os.path.join(CKPT_DIR, "jit_rcdm_step*.pt")))
latest = ckpts[-1] if ckpts else os.path.join(CKPT_DIR, "jit_rcdm_final.pt")
print(f"Sampling from: {latest}")

TEST_IMAGES = sorted(glob.glob("/content/drive/MyDrive/jit_rcdm/test_images/*.png"))
assert TEST_IMAGES, "Upload .png test images to Drive at MyDrive/jit_rcdm/test_images/"
cond_args = " ".join(TEST_IMAGES)
print(f"Conditioning images: {len(TEST_IMAGES)}")

!python scripts/sampling.py \
    --checkpoint     {latest} \
    --cond_images    {cond_args} \
    --out_dir        {SAMPLE_DIR} \
    --n_samples      4 \
    --num_steps      50 \
    --cfg_scale      1.0 \
    --wandb_project  jit-rcdm \
    --wandb_run_name S16-samples \
    --device         cuda

Sampling from: /content/drive/MyDrive/jit_rcdm/checkpoints/jit_rcdm_step0100000.pt
Conditioning images: 5
Loading JiT-RCDM from /content/drive/MyDrive/jit_rcdm/checkpoints/jit_rcdm_step0100000.pt...
  [EMA] loaded EMA weights for inference
  image_size=224, hidden_dim=384, cond_dim=128, h_dim=384, trained_steps=100000
Loading DinoV3 encoder...
Loading weights: 100% 211/211 [00:00<00:00, 1038.24it/s, Materializing param=norm.weight]                     
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: severinle (severinle-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /content/jit_rcdm/wandb/run-20260517_092953-kduj6cjn
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run S16-samples
wandb: ⭐️ Vie

## 11 — Monitor

In [13]:
!nvidia-smi
!ls -lh /content/drive/MyDrive/jit_rcdm/checkpoints/ 2>/dev/null || echo "No checkpoints yet"

Sun May 17 09:32:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             43W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

---

## Drive layout

```
MyDrive/jit_rcdm/
    dinov3_vits16_tmp/     ← upload before running
        config.json
        model.safetensors
    train_packed.pt        ← run pack_dataset.py locally, then upload (~150 MB)
    test_images/           ← upload before sampling
    checkpoints/           ← auto-created during training
    samples/               ← auto-created during sampling
```